In [20]:
# ==========================================
# 0. INSTALL & IMPORTS
# ==========================================
!pip install -q medmnist pillow torchvision

import torch
import medmnist
from medmnist import INFO
from torch.utils.data import Dataset, ConcatDataset
from torchvision import transforms
from PIL import Image
import os

# ==========================================
# 1. GLOBAL LABEL SPACES
# ==========================================
TOTAL_DISEASE_DIM = 60

ANATOMY_LABELS = {
    "pathology": 0,
    "breast": 1,
    "eye": 2,
    "lung": 3,
    "bone": 4,
    "brain": 5,
    "skin": 6,
    "blood": 7,
    "abdomen": 8
}

MODALITY_MAP = {
    "xray": 0,
    "ct": 1,
    "mri": 2,
    "fundus": 3,
    "dermoscopy": 4,
    "mammography": 5,
    "microscopy": 6
}

# ==========================================
# 2. DATASET CONFIG (REALISTIC + SAFE)
# ==========================================
MED_CONFIG = {
    "pathmnist": {
        "anatomy": "pathology", "modality": "microscopy",
        "off": 0, "cls": 9
    },
    "breastmnist": {
        "anatomy": "breast", "modality": "mammography",
        "off": 10, "cls": 2
    },
    "retinamnist": {
        "anatomy": "eye", "modality": "fundus",
        "off": 12, "cls": 5
    },
    "chestmnist": {
        "anatomy": "lung", "modality": "xray",
        "off": 17, "cls": 14
    },
    "fracturemnist": {
        "anatomy": "bone", "modality": "xray",
        "off": 31, "cls": 3
    },
    "tpmmnist": {
        "anatomy": "brain", "modality": "microscopy",
        "off": 34, "cls": 2
    },
    "dermamnist": {
        "anatomy": "skin", "modality": "dermoscopy",
        "off": 36, "cls": 7
    },
    "bloodmnist": {
        "anatomy": "blood", "modality": "microscopy",
        "off": 43, "cls": 8
    },
    "organamnist": {
        "anatomy": "abdomen", "modality": "ct",
        "off": 51, "cls": 11
    }
}

# ==========================================
# 3. UNIFIED DATASET WRAPPER
# ==========================================
class UnifiedMedMNIST(Dataset):
    def __init__(self, name, cfg):
        DataClass = getattr(medmnist, INFO[name]["python_class"])
        self.ds = DataClass(split="train", download=True, size=28)
        self.cfg = cfg

        self.transform = transforms.Compose([
            transforms.Resize((224, 224),interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
        ])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img, target = self.ds[idx]

        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)

        img = self.transform(img.convert("RGB"))
        target = target.flatten()

        disease_vec = torch.zeros(TOTAL_DISEASE_DIM)

        for i, val in enumerate(target):
            if len(target) == 1 or val == 1:
                disease_vec[self.cfg["off"] + i] = 1.0

        return {
            "image": img,
            "anatomy": ANATOMY_LABELS[self.cfg["anatomy"]],
            "modality": MODALITY_MAP[self.cfg["modality"]],
            "disease_label": disease_vec
        }

# ==========================================
# 4. BUILD MASTER DATASET (BULLETPROOF)
# ==========================================
def build_master_dataset():
    datasets = []
    available = set(INFO.keys())

    print("\nLoading MedMNIST datasets:\n")

    for name, cfg in MED_CONFIG.items():
        if name not in available:
            print(f"✗ {name}: not registered in MedMNIST")
            continue

        try:
            print(f"✓ {name}")
            datasets.append(UnifiedMedMNIST(name, cfg))
        except Exception as e:
            print(f"✗ {name}: failed ({str(e)[:60]})")

    assert len(datasets) > 0, "No datasets loaded — check MedMNIST install"
    return ConcatDataset(datasets)

# ==========================================
# 5. EXECUTION TEST
# ==========================================
if __name__ == "__main__":
    master_ds = build_master_dataset()

    print("\n===================================")
    print(f"Unified Dataset Size: {len(master_ds)}")

    sample = master_ds[0]
    print(
        f"Sample → Anatomy ID: {sample['anatomy']} | "
        f"Modality ID: {sample['modality']} | "
        f"Disease Vector Sum: {sample['disease_label'].sum().item()}"
    )



Loading MedMNIST datasets:

✓ pathmnist
✓ breastmnist
✓ retinamnist
✓ chestmnist
✗ fracturemnist: not registered in MedMNIST
✗ tpmmnist: not registered in MedMNIST
✓ dermamnist
✓ bloodmnist
✓ organamnist

Unified Dataset Size: 223617
Sample → Anatomy ID: 0 | Modality ID: 6 | Disease Vector Sum: 1.0


In [21]:
master_ds[2]

{'image': tensor([[[0.7647, 0.7647, 0.7647,  ..., 0.6627, 0.6706, 0.6706],
          [0.7569, 0.7569, 0.7569,  ..., 0.6627, 0.6706, 0.6706],
          [0.7569, 0.7569, 0.7569,  ..., 0.6627, 0.6706, 0.6706],
          ...,
          [0.6941, 0.6941, 0.6941,  ..., 0.7490, 0.7490, 0.7490],
          [0.6941, 0.6941, 0.6941,  ..., 0.7490, 0.7490, 0.7490],
          [0.6941, 0.6941, 0.6941,  ..., 0.7490, 0.7490, 0.7490]],
 
         [[0.3255, 0.3255, 0.3255,  ..., 0.1843, 0.1843, 0.1843],
          [0.3255, 0.3255, 0.3255,  ..., 0.1843, 0.1843, 0.1843],
          [0.3255, 0.3255, 0.3255,  ..., 0.1843, 0.1843, 0.1843],
          ...,
          [0.2157, 0.2157, 0.2078,  ..., 0.2157, 0.2157, 0.2157],
          [0.2157, 0.2157, 0.2078,  ..., 0.2157, 0.2157, 0.2157],
          [0.2157, 0.2157, 0.2078,  ..., 0.2157, 0.2157, 0.2157]],
 
         [[0.6078, 0.6078, 0.6078,  ..., 0.5059, 0.5059, 0.5059],
          [0.6078, 0.6078, 0.6078,  ..., 0.5059, 0.5059, 0.5059],
          [0.6078, 0.6078, 0.60

In [22]:
import torch.nn as nn
from torchvision import models

In [23]:
class GlobalMedicalNavigator(nn.Module):
    def __init__(self, num_anatomy=10, num_modality=7, embedding_dim=512):
        super(GlobalMedicalNavigator, self).__init__()

        # 1. SHARED VISUAL BACKBONE
        # DenseNet121 is the industry standard for medical imaging due to its
        # ability to reuse features across different scales.
        backbone = models.densenet121(weights='IMAGENET1K_V1')
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # DenseNet121 final feature dimension is 1024
        self.feature_dim = 1024

        # 2. EMBEDDING HEAD (The "Visual Fingerprint" z)
        # Purpose: KNN / Retrieval Engine logic.
        # This creates the "Neighborhood" clusters.
        self.embedding_head = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, embedding_dim),
            nn.LayerNorm(embedding_dim) # Normalization helps KNN search stability
        )

        # 3. ANATOMY HEAD (Body Part Identifier)
        # Purpose: Routing/Gating decision.
        self.anatomy_head = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_anatomy)
        )

        # 4. MODALITY HEAD (Scan Type Identifier)
        # Purpose: Sanity Check / Routing decision.
        self.modality_head = nn.Sequential(
            nn.Linear(self.feature_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_modality)
        )

    def forward(self, x):
        # Extract Feature Maps
        # These are kept for Grad-CAM (Step 5)
        feature_maps = self.features(x)

        # Global Average Pooling
        pooled = self.pool(feature_maps)
        flattened = torch.flatten(pooled, 1)

        # Branching into the Multi-Head signals
        z = self.embedding_head(flattened)
        anatomy_logits = self.anatomy_head(flattened)
        modality_logits = self.modality_head(flattened)

        return {
            "z": z,                      # For KNN
            "anatomy_probs": anatomy_logits, # For Routing
            "modality_probs": modality_logits, # For Routing
            "feature_maps": feature_maps     # For Grad-CAM
        }

In [24]:
class NavigatorLoss(nn.Module):
    """
    Simultaneously optimizes Anatomy and Modality.
    Weights can be adjusted to favor one head over the other.
    """
    def __init__(self):
        super(NavigatorLoss, self).__init__()
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, outputs, targets):
        loss_anatomy = self.criterion(outputs['anatomy_probs'], targets['anatomy'])
        loss_modality = self.criterion(outputs['modality_probs'], targets['modality'])

        # Combine losses (1:1 weighting)
        # We don't train on disease_labels here; that's for the Experts.
        total_loss = loss_anatomy + loss_modality
        return total_loss, loss_anatomy, loss_modality

In [26]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

# ==========================================
# 1. SETUP: DEVICE & DATA
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# master_ds is the ConcatDataset we created in Step 1
train_loader = DataLoader(master_ds, batch_size=64, shuffle=True, num_workers=2,drop_last=True)

# Initialize Model
model = GlobalMedicalNavigator(num_anatomy=10, num_modality=7).to(device)

# Initialize Optimizer & Loss
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = NavigatorLoss() # The loss we defined in the previous step

# ==========================================
# 2. THE TRAINING FUNCTION
# ==========================================
def train_navigator(model, loader, optimizer, criterion, epochs=2):
    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        correct_anatomy = 0
        correct_modality = 0
        total = 0

        # Progress bar for tracking
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            # Move data to GPU
            images = batch['image'].to(device)
            target_anatomy = batch['anatomy'].to(device)
            target_modality = batch['modality'].to(device)

            # Forward Pass
            optimizer.zero_grad()
            outputs = model(images)

            # Calculate Multi-Task Loss
            targets = {'anatomy': target_anatomy, 'modality': target_modality}
            loss, loss_a, loss_m = criterion(outputs, targets)

            # Backward Pass
            loss.backward()
            optimizer.step()

            # Metrics Calculation
            running_loss += loss.item()

            # Anatomy Accuracy
            _, pred_a = torch.max(outputs['anatomy_probs'], 1)
            correct_anatomy += (pred_a == target_anatomy).sum().item()

            # Modality Accuracy
            _, pred_m = torch.max(outputs['modality_probs'], 1)
            correct_modality += (pred_m == target_modality).sum().item()

            total += target_anatomy.size(0)

            # Update Progress Bar
            pbar.set_postfix({
                'Loss': f"{loss.item():.4f}",
                'Ana_Acc': f"{100 * correct_anatomy / total:.1f}%",
                'Mod_Acc': f"{100 * correct_modality / total:.1f}%"
            })

        print(f"\nEpoch {epoch+1} Summary: Loss: {running_loss/len(loader):.4f}")

# ==========================================
# 3. RUN TRAINING
# ==========================================
if __name__ == "__main__":
    # Start training
    train_navigator(model, train_loader, optimizer, criterion, epochs=2)

    # SAVE THE BACKBONE
    # This is critical! We will need this frozen backbone for Step 3 & 4.
    torch.save(model.state_dict(), "global_navigator.pth")
    print("Backbone saved successfully.")

Epoch 1/2: 100%|██████████| 3494/3494 [37:57<00:00,  1.53it/s, Loss=0.0000, Ana_Acc=99.9%, Mod_Acc=99.9%]



Epoch 1 Summary: Loss: 0.0154


Epoch 2/2: 100%|██████████| 3494/3494 [37:55<00:00,  1.54it/s, Loss=0.0004, Ana_Acc=100.0%, Mod_Acc=100.0%]


Epoch 2 Summary: Loss: 0.0030
Backbone saved successfully.


In [27]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for your project
import os
save_path = "/content/drive/MyDrive/Medical_AI_Project"
if not os.path.exists(save_path):
    os.makedirs(save_path)

Mounted at /content/drive


In [28]:
import pickle
import torch

# 1. Save the Model Weights
torch.save(model.state_dict(), f"{save_path}/global_navigator.pth")

# 2. Save the Label Mappings and Configs
# This is crucial so Anatomy '0' stays 'pathology' in the next notebook
navigator_metadata = {
    "ANATOMY_LABELS": ANATOMY_LABELS,
    "MODALITY_MAP": MODALITY_MAP,
    "MED_CONFIG": MED_CONFIG,
    "TOTAL_DISEASE_DIM": TOTAL_DISEASE_DIM
}

with open(f"{save_path}/navigator_metadata.pkl", "wb") as f:
    pickle.dump(navigator_metadata, f)

print(f"Model and Metadata saved to: {save_path}")

Model and Metadata saved to: /content/drive/MyDrive/Medical_AI_Project


In [29]:
import shutil

# Local MedMNIST location in Colab
local_medmnist = os.path.expanduser("~/.medmnist")
drive_medmnist = f"{save_path}/medmnist_data"

if not os.path.exists(drive_medmnist):
    os.makedirs(drive_medmnist)

# Copy all .npz files to your Drive
for file in os.listdir(local_medmnist):
    if file.endswith(".npz"):
        shutil.copy(os.path.join(local_medmnist, file), drive_medmnist)

print(f"Dataset files (.npz) backed up to: {drive_medmnist}")

Dataset files (.npz) backed up to: /content/drive/MyDrive/Medical_AI_Project/medmnist_data
